# Le rapport de l'or à la monnaie aux États-Unis · *US gold relative to money*

Notebook compagnon de l'enquête **L'or monte-t-il parce que les monnaies s'effondrent ?** — [lire l'article](https://nmlab.io/ressources/prix-de-l-or-et-effondrement-des-monnaies).
Companion notebook to the study **Is gold rising because currencies are collapsing?**.

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données publiques du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's public data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


import io
import re
import urllib.request
from functools import lru_cache

import numpy as np
import pandas as pd
from pandas import DataFrame, Series

CMO_PAGE = "https://www.worldbank.org/en/research/commodity-markets"
CMO_FILE = ("https://thedocs.worldbank.org/en/doc/74e8be41ceb20fa0da750cda2f6b9e4e-0050012026"
            "/related/CMO-Historical-Data-Monthly.xlsx")
FRED_CSV = "https://fred.stlouisfed.org/graph/fredgraph.csv?id={}"

# H.10 : EUR, GBP et AUD sont cotés en dollars par unité étrangère — on les inverse
# pour obtenir partout des unités locales par dollar, comme dans l'article.
# H.10 quotes EUR, GBP and AUD as dollars per foreign unit: invert them so every
# series is local units per dollar, as in the article.
FX = {"EUR": ("DEXUSEU", True), "JPY": ("DEXJPUS", False), "GBP": ("DEXUSUK", True),
      "CHF": ("DEXSZUS", False), "CAD": ("DEXCAUS", False), "AUD": ("DEXUSAL", True),
      "CNY": ("DEXCHUS", False)}


def _fetch(url: str, tries: int = 4) -> bytes:
    """Télécharge une URL, avec quelques reprises (FRED coupe parfois la connexion).
    Download a URL, retrying a few times (FRED occasionally drops the connection)."""
    import time
    for attempt in range(tries):
        try:
            return urllib.request.urlopen(url, timeout=90).read()
        except Exception:
            if attempt == tries - 1:
                raise
            time.sleep(2 * (attempt + 1))
    raise RuntimeError("unreachable")


@lru_cache(maxsize=None)
def load_gold_usd() -> Series:
    """Or en dollars par once, moyennes mensuelles depuis 1960.

    Source : « Commodity Price Data » (Pink Sheet) de la Banque mondiale, feuille
    « Monthly Prices », colonne Gold — le fixing de Londres, en accès libre.
    World Bank Pink Sheet, monthly London gold price in US dollars per troy ounce.
    """
    try:
        raw = _fetch(CMO_FILE)
    except Exception:                                  # millésime renouvelé : on relit le lien
        page = _fetch(CMO_PAGE).decode("utf-8", "ignore")
        link = re.search(r"https://[^\"']*CMO-Historical-Data-Monthly\.xlsx", page)
        raw = _fetch(link.group(0))
    table = pd.read_excel(io.BytesIO(raw), sheet_name="Monthly Prices", skiprows=4)
    table = table.rename(columns={table.columns[0]: "date"})[["date", "Gold"]].dropna()
    dates = pd.to_datetime(table["date"].str.replace("M", "-"), format="%Y-%m")
    return Series(table["Gold"].values, index=dates).astype(float)


@lru_cache(maxsize=None)
def load_fred(series_id: str) -> Series:
    """Série FRED (CSV public, sans clé) ramenée à des moyennes mensuelles.
    A FRED series (public CSV, no key) averaged to monthly frequency."""
    table = pd.read_csv(io.StringIO(_fetch(FRED_CSV.format(series_id)).decode()))
    values = pd.to_numeric(table[table.columns[1]], errors="coerce")
    series = Series(values.values, index=pd.to_datetime(table[table.columns[0]])).dropna()
    return series.resample("MS").mean()


def load_gold_in_currencies(start: str, end: str) -> DataFrame:
    """Prix de l'or dans les huit devises du panier, mois par mois.

    Chaque prix local est le produit de la moyenne mensuelle de l'or en dollars
    et de la moyenne mensuelle du taux de change — l'ordre des opérations retenu
    par l'article. Le dollar vaut 1 par construction.
    Gold priced in the eight basket currencies, month by month.
    """
    gold = load_gold_usd()
    prices = {"USD": gold}
    for code, (series_id, invert) in FX.items():
        rate = load_fred(series_id)
        prices[code] = gold * (1 / rate if invert else rate)
    return DataFrame(prices).loc[start:end].dropna()


def effective_index(prices: DataFrame) -> Series:
    """Indice or effectif : moyenne géométrique équipondérée des huit prix locaux,
    base 100 au premier mois. Seule la moyenne géométrique garantit que l'indice
    des devises mesurées contre l'or est exactement l'inverse de celui-ci.
    Equal-weighted geometric mean of the eight local prices, first month = 100.
    """
    return 100 * np.exp(np.log(prices / prices.iloc[0]).mean(axis=1))


PARITY, PARITY_END = 35.0, "1968-03-01"   # parité officielle jusqu'au London Gold Pool
END = "2026-02-01"                        # borne de l'article ; None = jusqu'à aujourd'hui


def load_gold_over_m2(end: str | None = END) -> Series:
    """Rapport « or ÷ M2 » aux États-Unis, mensuel depuis 1959.

    Avant avril 1968, l'or est porté à sa parité officielle de 35 dollars l'once ;
    ensuite vient le marché londonien. Le raccord est volontairement visible : la
    partie ancienne est un contrôle historique, pas un prix de marché moderne.
    US gold-to-M2 ratio: official 35-dollar parity until March 1968, London market after.
    """
    market = load_gold_usd()
    official = Series(PARITY, index=pd.date_range("1959-01-01", PARITY_END, freq="MS"))
    gold = pd.concat([official, market.loc["1968-04-01":]])
    ratio = (gold / load_fred("M2SL")).dropna()
    return ratio.loc[:end] if end else ratio


from matplotlib.figure import Figure
from matplotlib.ticker import FixedLocator, FuncFormatter

LABELS = {
    "fr": dict(
        title="Un ancrage très lâche : le rapport de l'or à la monnaie",
        sub="Prix de l'once divisé par M2, États-Unis, échelle logarithmique — contrôle séparé du panier.",
        peak="Sommet de janvier 1980", trough="Creux d'avril 2001", div="divisé par {:.0f}",
        parity="parité officielle\n(35 dollars l'once)",
        note="Le niveau du rapport dépend des unités retenues (dollars par once, milliards de dollars) : seules ses\n"
             "variations s'interprètent. Sources : Banque mondiale (or) ; Réserve fédérale, M2SL."),
    "en": dict(
        title="A very loose anchor: gold relative to money",
        sub="Ounce price divided by M2, United States, logarithmic scale — a control separate from the basket.",
        peak="January 1980 peak", trough="April 2001 trough", div="divided by {:.0f}",
        parity="official parity\n(35 dollars an ounce)",
        note="The level of the ratio depends on the units used (dollars per ounce, billions of dollars): only its\n"
             "variations can be read. Sources: World Bank (gold); Federal Reserve, M2SL."),
}


def build_figure(ratio: Series, lang: str) -> Figure:
    """Courbe logarithmique du rapport, sommet et creux annotés, période de parité grisée."""
    text = LABELS[lang]
    peak, trough = pd.Timestamp("1980-01-01"), pd.Timestamp("2001-04-01")

    fig = nm.figure(height_px=1080)
    ax = nm.axes(fig, left=0.075, right=0.982)
    ax.axvspan(ratio.index[0], pd.Timestamp(PARITY_END), color=nm.COLORS["card"], zorder=1)
    ax.plot(ratio.index, ratio, color=nm.COLORS["amber"], lw=3.0, zorder=4)
    ax.set_yscale("log")
    ax.yaxis.set_major_locator(FixedLocator([0.05, 0.1, 0.2, 0.3, 0.4]))
    ax.yaxis.set_minor_locator(FixedLocator([]))
    ax.yaxis.set_major_formatter(FuncFormatter(
        lambda v, _: f"{v:.2f}".replace(".", "," if lang == "fr" else ".")))

    ax.text(pd.Timestamp("1959-06-01"), ratio.max() * 0.92, text["parity"], ha="left", va="top",
            fontsize=18.5, color=nm.COLORS["muted"], linespacing=1.4, zorder=5)
    ax.set_ylim(ratio.min() * 0.74, ratio.max() * 1.16)     # place pour les annotations
    for date, key, days, offset, align in ((peak, "peak", 760, 1.42, "left"),
                                           (trough, "trough", 420, 0.80, "left")):
        ax.plot([date], [ratio[date]], "o", ms=13, color=nm.COLORS["rose"], zorder=6)
        ax.annotate(text[key], xy=(date, ratio[date]),
                    xytext=(date + pd.Timedelta(days=days), ratio[date] * offset),
                    fontsize=20.5, color=nm.COLORS["rose"], fontweight="bold", va="center",
                    ha=align,
                    arrowprops=dict(arrowstyle="-", color=nm.COLORS["rose"], lw=1.8, alpha=0.75))
    mid = peak + (trough - peak) / 2
    ax.annotate("", xy=(mid, ratio[peak]), xytext=(mid, ratio[trough]),
                arrowprops=dict(arrowstyle="<->", color=nm.COLORS["muted"], lw=2))
    ax.text(mid + pd.Timedelta(days=260), (ratio[peak] * ratio[trough]) ** 0.5,
            text["div"].format(ratio[peak] / ratio[trough]), ha="left", va="center",
            fontsize=21, fontweight="bold", color=nm.COLORS["text"])

    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig


build_figure(load_gold_over_m2(), LANG)